# 공장 센서 데이터 설비 상태 분류 프로젝트

이 프로젝트는 1일차 파이썬 문법 수업을 마무리하기 위한 미니 프로젝트입니다.

가상의 반도체 공장에서 수집한 설비 센서 데이터를 이용해 설비 상태를 다음 세 가지 중 하나로 분류합니다.

- `정상`: 현재 센서값이 안정적인 설비
- `점검`: 당장 멈출 정도는 아니지만 점검이 필요한 설비
- `위험`: 이상 징후가 커서 빠른 확인이 필요한 설비

이 프로젝트에서는 `pandas`, `numpy`, `scikit-learn` 같은 라이브러리를 사용하지 않습니다.
리스트, 인덱싱, 반복문, 조건문, 함수만 사용해 간단한 분류기의 동작 원리를 직접 구현합니다.


## 프로젝트 상황

공장에는 여러 설비가 있고, 각 설비에서는 다음과 같은 센서값이 기록됩니다.

| 컬럼 | 의미 | 예시 |
|---|---|---|
| `temperature` | 설비 온도 | 67.2 |
| `pressure` | 공정 압력 | 1.03 |
| `vibration` | 진동 수준 | 1.85 |
| `gas_flow` | 가스 흐름량 | 121.5 |
| `cycle_time` | 1회 생산 사이클 시간 | 31.2 |
| `status` | 설비 상태 정답 | 정상 / 점검 / 위험 |

목표는 센서값 5개를 입력받아 설비 상태를 예측하는 것입니다.


## 데이터

아래 데이터는 1차원 리스트 `D`에 저장되어 있습니다.

원래는 한 줄에 하나의 설비 데이터가 있다고 생각하면 됩니다.

```text
temperature, pressure, vibration, gas_flow, cycle_time, status
67.2, 1.03, 1.85, 121.5, 31.2, 정상
...
```

하지만 이번 프로젝트에서는 파일 처리나 pandas 없이 파이썬 리스트만 사용하기 위해 데이터를 1차원 리스트로 넣어두었습니다.


In [12]:
D = [
    'temperature',    'pressure',    'vibration',    'gas_flow',    'cycle_time',    'status',
    70.7,    1.07,    1.65,    112.7,    30.0,    '정상',
    63.9,    1.04,    1.29,    121.8,    29.6,    '정상',
    60.8,    1.09,    2.04,    130.5,    30.6,    '정상',
    63.7,    1.01,    0.83,    126.9,    28.5,    '정상',
    61.2,    1.03,    2.34,    123.2,    28.7,    '정상',
    62.7,    1.08,    1.52,    118.2,    30.1,    '정상',
    67.4,    1.16,    0.98,    118.9,    29.1,    '정상',
    57.3,    0.98,    1.09,    127.6,    30.3,    '정상',
    59.8,    1.08,    1.42,    113.3,    29.4,    '정상',
    63.1,    0.99,    1.61,    110.8,    30.0,    '위험',
    68.2,    1.09,    2.15,    127.6,    33.2,    '정상',
    61.4,    1.09,    0.67,    119.0,    27.1,    '정상',
    68.4,    1.12,    1.5,    130.9,    29.3,    '정상',
    65.5,    1.05,    1.07,    124.2,    34.0,    '정상',
    61.7,    1.09,    1.52,    130.8,    31.4,    '정상',
    67.2,    1.08,    1.72,    115.0,    33.0,    '정상',
    60.6,    1.05,    1.91,    117.8,    30.8,    '정상',
    68.7,    1.04,    1.16,    107.4,    28.8,    '정상',
    63.1,    1.01,    1.12,    123.9,    29.5,    '정상',
    64.6,    1.08,    1.17,    120.5,    34.4,    '정상',
    67.8,    0.96,    1.56,    124.8,    32.7,    '정상',
    65.5,    1.01,    1.23,    125.2,    31.1,    '정상',
    63.3,    1.03,    1.39,    125.6,    29.3,    '정상',
    66.6,    1.09,    1.84,    117.8,    32.9,    '정상',
    62.8,    1.07,    1.22,    113.3,    30.9,    '정상',
    65.9,    1.04,    1.74,    118.8,    29.2,    '정상',
    64.5,    1.15,    2.01,    119.4,    31.5,    '정상',
    63.5,    1.01,    2.03,    119.4,    27.5,    '정상',
    62.1,    1.05,    1.66,    128.2,    29.2,    '정상',
    67.3,    1.07,    1.53,    126.2,    31.4,    '정상',
    62.2,    1.1,    0.99,    122.3,    30.5,    '정상',
    66.4,    1.0,    1.58,    121.3,    32.3,    '정상',
    62.5,    1.04,    1.47,    115.5,    35.3,    '정상',
    64.8,    0.95,    2.3,    116.6,    31.2,    '정상',
    67.8,    1.0,    1.38,    119.3,    30.3,    '정상',
    59.8,    0.97,    1.53,    127.2,    31.6,    '위험',
    70.5,    1.08,    1.34,    121.4,    26.8,    '정상',
    61.9,    0.98,    1.74,    125.2,    29.5,    '정상',
    63.1,    0.95,    1.54,    123.2,    31.3,    '정상',
    62.5,    1.16,    2.2,    121.5,    29.6,    '정상',
    70.8,    0.89,    3.19,    115.9,    38.0,    '점검',
    77.2,    0.97,    3.12,    115.3,    35.0,    '점검',
    69.8,    1.03,    2.86,    124.9,    35.1,    '점검',
    73.0,    0.97,    3.22,    115.4,    35.4,    '점검',
    70.2,    0.87,    2.71,    122.8,    35.6,    '점검',
    78.5,    0.67,    3.21,    116.4,    35.2,    '점검',
    76.2,    0.93,    2.99,    115.6,    43.3,    '점검',
    73.0,    0.9,    2.87,    113.8,    32.6,    '점검',
    71.2,    1.08,    4.0,    116.8,    38.9,    '점검',
    79.2,    0.93,    4.75,    119.4,    36.4,    '점검',
    78.5,    0.69,    2.77,    114.6,    33.6,    '점검',
    71.4,    0.83,    2.71,    129.3,    37.6,    '점검',
    71.8,    0.99,    3.02,    118.3,    36.9,    '점검',
    78.2,    0.77,    3.77,    113.7,    33.0,    '위험',
    76.0,    0.81,    4.41,    116.1,    37.8,    '점검',
    72.3,    0.95,    1.8,    113.4,    34.4,    '점검',
    73.1,    0.88,    3.66,    108.0,    40.8,    '점검',
    80.8,    0.86,    3.19,    114.7,    39.3,    '점검',
    79.2,    0.78,    2.74,    106.6,    33.1,    '점검',
    65.8,    0.91,    2.33,    122.0,    38.4,    '점검',
    68.1,    0.8,    2.74,    121.4,    33.8,    '점검',
    66.3,    0.88,    3.48,    109.1,    40.2,    '점검',
    72.9,    0.92,    2.56,    112.8,    35.6,    '점검',
    63.4,    0.9,    1.85,    112.8,    32.2,    '점검',
    79.7,    0.91,    3.85,    118.5,    32.6,    '점검',
    63.3,    0.85,    3.87,    113.5,    38.0,    '점검',
    75.6,    0.76,    3.42,    124.8,    34.5,    '점검',
    64.3,    0.86,    2.85,    115.5,    33.7,    '점검',
    72.3,    0.7,    3.84,    122.5,    39.4,    '점검',
    66.2,    0.91,    0.96,    114.7,    35.8,    '점검',
    68.2,    0.91,    3.3,    118.6,    36.3,    '점검',
    72.9,    0.72,    2.33,    110.1,    36.8,    '점검',
    72.0,    1.0,    2.55,    111.6,    36.7,    '점검',
    71.1,    0.92,    2.77,    121.5,    35.2,    '점검',
    76.2,    0.88,    4.32,    104.6,    37.3,    '점검',
    70.4,    0.9,    4.09,    106.2,    32.6,    '점검',
    68.2,    0.9,    1.78,    113.6,    32.7,    '점검',
    67.7,    0.85,    3.49,    109.2,    34.7,    '점검',
    79.8,    0.69,    2.66,    100.0,    43.8,    '위험',
    84.9,    0.92,    5.83,    103.6,    40.1,    '점검',
    80.4,    0.8,    6.4,    96.8,    44.3,    '위험',
    83.7,    0.91,    5.44,    99.2,    41.1,    '위험',
    78.4,    0.65,    5.99,    110.5,    46.0,    '위험',
    85.4,    0.67,    5.56,    100.3,    43.7,    '위험',
    85.5,    0.72,    3.43,    101.2,    41.0,    '위험',
    73.2,    0.64,    6.11,    114.8,    41.8,    '위험',
    84.5,    0.84,    4.1,    102.7,    43.3,    '위험',
    80.8,    0.56,    4.87,    110.4,    47.9,    '위험',
    88.2,    0.69,    6.98,    107.7,    38.7,    '위험',
    83.4,    0.68,    5.69,    102.3,    47.7,    '위험',
    81.2,    0.69,    6.12,    104.8,    42.1,    '위험',
    85.7,    0.68,    4.59,    99.2,    38.4,    '위험',
    77.5,    0.91,    4.34,    103.1,    39.6,    '위험',
    77.2,    0.87,    6.05,    98.7,    43.5,    '위험',
    76.3,    0.64,    4.33,    96.7,    44.6,    '점검',
    82.7,    0.77,    5.04,    93.5,    43.3,    '위험',
    75.8,    0.81,    5.73,    108.2,    42.3,    '위험',
    71.9,    0.82,    6.37,    92.6,    44.0,    '점검',
    85.4,    0.61,    6.88,    100.6,    39.7,    '위험',
    86.2,    0.75,    4.58,    109.2,    36.7,    '위험',
    88.2,    0.64,    5.47,    100.2,    45.0,    '위험',
    84.0,    0.64,    5.95,    100.9,    41.1,    '위험',
    79.0,    0.62,    4.44,    109.6,    43.0,    '위험',
    85.1,    0.76,    4.23,    89.7,    41.1,    '위험',
    81.2,    0.72,    4.41,    110.1,    37.4,    '위험',
    74.3,    0.73,    7.28,    105.9,    40.0,    '위험',
    82.7,    0.79,    6.09,    100.1,    40.7,    '위험',
    80.5,    0.76,    6.72,    117.6,    40.9,    '위험',
    77.6,    0.71,    4.96,    106.4,    45.2,    '위험',
    81.2,    0.85,    5.05,    95.6,    38.0,    '위험',
]

## Mission 1. 데이터 전처리

1차원 리스트 `D`에 저장된 데이터를 다음처럼 나누어 저장하세요.

- 센서값 5개는 2차원 리스트 `X`에 저장
- 설비 상태 정답은 1차원 리스트 `y`에 저장

예상 형태는 다음과 같습니다.

```python
X = [
    [70.7, 1.07, 1.65, 112.7, 30.0],
    [63.9, 1.02, 1.42, 121.6, 29.4],
    ...
]

y = ['정상', '정상', ...]
```


In [35]:
# X = []
# y = []

# # 데이터는 6개씩 한 묶음입니다.
# # temperature, pressure, vibration, gas_flow, cycle_time, status
# for i in range(0, len(D), 6):  ## None을 적절한 코드로 바꾸세요. ##
#     d = D[i:i+6]               ## None을 적절한 코드로 바꾸세요. ##
#     X.append(d[:5])            ## None을 적절한 코드로 바꾸세요. ##
#     y.append(d[-1])             ## None을 적절한 코드로 바꾸세요. ##

# # 첫 번째 묶음은 컬럼명이므로 제외합니다.
# X = X[1:]
# y = y[1:]

# print('센서 데이터 개수:', len(X))
# print('정답 데이터 개수:', len(y))
# print('첫 번째 센서 데이터:', X[0])
# print('첫 번째 정답:', y[0])

###################################################
# numpy version
###################################################
import numpy as np

D_np = np.array(D[6:]).reshape(-1, 6)
X = D_np[:, :5].astype(float)
y = D_np[:, -1]

print('센서 데이터 개수:', len(X))
print('정답 데이터 개수:', len(y))
print('첫 번째 센서 데이터:', X[0])
print('첫 번째 정답:', y[0])

센서 데이터 개수: 110
정답 데이터 개수: 110
첫 번째 센서 데이터: [ 70.7    1.07   1.65 112.7   30.  ]
첫 번째 정답: 정상


## Mission 2. 설비 상태 점수 계산하기

선형 분류기는 각 상태에 대해 점수를 계산합니다.

```text
점수 = 온도×가중치 + 압력×가중치 + 진동×가중치 + 가스흐름×가중치 + 사이클시간×가중치 + 절편
```

한 설비에 대해 `정상`, `점검`, `위험` 점수 3개를 계산합니다.
점수가 가장 높은 상태가 모델의 예측 결과가 됩니다.


In [22]:
# def clf(X):
#     weight = [
#         [ -0.111107836, 0.285036031, -0.855377268, 0.131106216, -0.467163383 ],
#         [ -0.009777854, -0.03275751, 0.209529283, -0.036268899, 0.196702295  ],
#         [  0.12088569, -0.252278521, 0.645847985, -0.094837317, 0.270461088  ]
#     ]
#     intercept = [9.96210768, -0.815847958, -9.146259722]
#
#     # 모든 설비의 점수를 저장할 리스트입니다.
#     scores = []
#
#     # X에서 설비 하나의 센서값을 꺼냅니다.
#     for x in X:
#         # 설비 하나에 대한 정상/점검/위험 점수를 저장합니다.
#         model_score = []
#
#         # 상태가 3개이므로 3번 반복합니다.
#         for i in range(3):  ## None을 적절한 코드로 바꾸세요. ##
#             w = weight[i]
#             z = 0.0
#
#             # 센서값 5개와 가중치 5개를 곱해 더합니다.
#             for j in range(len(x)):
#                 z = z + w[j] * x[j]  ## 센서값과 가중치를 곱해서 더합니다. ##
#
#             # 절편을 더합니다.
#             z = z + intercept[i]  ## 절편을 더합니다. ##
#
#             model_score.append(z)
#
#         scores.append(model_score)
#     return scores


###################################################
# numpy version
###################################################
def clf(X):
    weight = np.array([
        [ -0.111107836, 0.285036031, -0.855377268, 0.131106216, -0.467163383 ],
        [ -0.009777854, -0.03275751, 0.209529283, -0.036268899, 0.196702295  ],
        [  0.12088569, -0.252278521, 0.645847985, -0.094837317, 0.270461088  ]
    ])
    intercept = np.array([9.96210768, -0.815847958, -9.146259722])

    scores = np.dot(X, weight.T) + intercept

    return scores

In [30]:
print(X.shape)
scores = clf(X)

print('첫 번째 설비의 상태별 점수:')
print(scores[0])

(110, 5)
첫 번째 설비의 상태별 점수:
[ 1.76116879  0.61709448 -2.37826327]


계산이 제대로 되었다면 첫 번째 설비의 점수는 대략 다음과 비슷하게 나옵니다.

```text
[1.761..., 0.617..., -2.378...]
```


## Mission 3. 가장 높은 점수의 상태 선택하기

모델이 계산한 점수 3개 중 가장 큰 값을 찾고, 그 위치를 이용해 예측 상태를 선택합니다.

예를 들어 점수가 아래처럼 나왔다면,

```python
[1.76, 0.61, -2.37]
```

가장 큰 값은 `1.76`이고 위치는 `0`입니다.
`STATUS[0]`은 `정상`이므로 모델의 예측 결과는 `정상`입니다.

`max(score)` 함수는 주어진 score 숫자들 중에 가장 큰 숫자를 찾고 `score.index(max_score)`는 max_score의 인덱스를 반환합니다.


In [36]:
# 첫 번째 설비 하나만 예측해봅니다.
# STATUS = ['정상', '점검', '위험']
# score = scores[0]
#
# max_score = max(score)
# max_index = score.index(max_score)       ## None을 적절한 코드로 바꾸세요. ##
# predicted_status = STATUS[max_index]    ## None을 적절한 코드로 바꾸세요. ##
# truth_status = y[0]             ## None을 적절한 코드로 바꾸세요. ##
#
# print('점수:', score)
# print('예측 상태:', predicted_status)
# print('실제 상태:', truth_status)

###################################################
# numpy version
###################################################
STATUS = np.array(['정상', '점검', '위험'])
truth_status = y[0]
predicted_status = STATUS[np.argmax(scores[0])]

print('점수:', scores[0])
print('예측 상태:', predicted_status)
print('실제 상태:', truth_status)

점수: [ 1.76116879  0.61709448 -2.37826327]
예측 상태: 정상
실제 상태: 정상


## Mission 4. 전체 설비 예측과 정확도 계산

이제 전체 설비에 대해 예측을 수행하고, 실제 정답과 비교해 정확도를 계산합니다.

이 데이터는 일부러 완벽하게 나누어지지 않도록 만들었습니다.
현장 센서 데이터에는 경계 구간이 있기 때문에 정확도는 100%가 아니라 90% 초반 정도가 나오도록 설계했습니다.


In [43]:
# correct = 0
# predictions = []
#
# for i in range(len(scores)):
#     score = scores[i]
#
#     max_score = max(score)
#     max_index = score.index(max_score)
#
#     predicted_status = STATUS[max_index]  ## None을 적절한 코드로 바꾸세요. ##
#     truth_status = y[i]           ## None을 적절한 코드로 바꾸세요. ##
#
#     predictions.append(predicted_status)
#
#     if predicted_status == truth_status:     ## None을 적절한 코드로 바꾸세요. ##
#         correct = correct + 1
#
# accuracy = correct / len(X) * 100
#
# print('전체 설비 수:', len(X))
# print('정답 개수:', correct)
# print(f'정확도: {accuracy:.1f}%')

###################################################
# numpy version
###################################################
predicted_status = STATUS[np.argmax(scores, axis=1)]
correct = np.sum( predicted_status == y )
accuracy = correct / len(X) * 100

print('전체 설비 수:', len(X))
print('정답 개수:', correct)
print(f'정확도: {accuracy:.1f}%')

전체 설비 수: 110
정답 개수: 101
정확도: 91.8%


## Bonus Mission. 위험 설비 경고 메시지 출력하기

시간이 남으면 예측 결과가 `위험`인 설비만 찾아 경고 메시지를 출력해보세요.


In [44]:
import time, random
from IPython.display import clear_output

# 학습에 사용한 데이터와는 다른 새로운 센서 데이터입니다.
# 데이터 순서: temperature, pressure, vibration, gas_flow, cycle_time
live_data = [
    [64.2, 1.05, 1.4, 122.0, 30.1],
    [71.5, 0.96, 2.8, 116.4, 34.2],
    [78.3, 0.84, 4.9, 103.7, 41.8],
    [66.1, 1.02, 1.9, 119.8, 31.0],
    [74.8, 0.91, 3.5, 111.2, 36.9],
    [82.0, 0.78, 5.7, 98.5, 45.2],
    [62.7, 1.08, 1.2, 126.3, 29.4],
    [70.9, 0.95, 2.6, 115.0, 33.8],
    [77.1, 0.86, 4.3, 106.8, 39.7],
    [65.5, 1.03, 1.6, 121.5, 30.6],
]

# 상태별 조치 메시지입니다.
action_message = {
    "정상": "🟢 정상 가동: 현재 설비 상태가 안정적입니다.",
    "점검": "🟡 점검 필요: 다음 생산 배치 전 설비 확인을 권장합니다.",
    "위험": "🔴 위험 신호: 즉시 설비를 확인하고 생산 중단 여부를 검토해야 합니다."
}

# for i in range(len(live_data)):
#     sensor = live_data[i]

#     # 설비 하나를 분류기에 넣기 위해 리스트 안에 한 번 더 넣습니다.
#     scores = clf([sensor])

#     # 첫 번째 설비의 점수만 꺼냅니다.
#     score = scores[0]

#     # 가장 높은 점수의 위치를 찾습니다.
#     max_score = max(score)
#     max_index = score.index(max_score)

#     # 예측 상태를 구합니다.
#     predicted_status = STATUS[max_index]

#     clear_output(wait=True)

#     print("실시간 설비 모니터링")
#     print("-" * 40)
#     print("설비 번호:", "EQP-" + str(i + 1).zfill(3))
#     print("센서 데이터:", sensor)
#     print("상태별 점수:", score)
#     print("예측 상태:", predicted_status)
#     print()
#     print(action_message[predicted_status])

#     time.sleep(1)


###################################################
# numpy version
###################################################
for i in range(len(live_data)):
    sensor = live_data[i]

    score = clf([sensor])
    predicted_status = STATUS[np.argmax(score)]
    clear_output(wait=True)

    print("실시간 설비 모니터링")
    print("-" * 40)
    print("설비 번호:", "EQP-" + str(i + 1).zfill(3))
    print("센서 데이터:", sensor)
    print("상태별 점수:", score)
    print("예측 상태:", predicted_status)
    print()
    print(action_message[predicted_status])

    time.sleep(1)


실시간 설비 모니터링
----------------------------------------
설비 번호: EQP-010
센서 데이터: [65.5, 1.03, 1.6, 121.5, 30.6]
상태별 점수: [[ 3.24373363  0.45762822 -3.70136185]]
예측 상태: 정상

🟢 정상 가동: 현재 설비 상태가 안정적입니다.


## 마무리

이번 프로젝트에서는 파이썬 문법 프로젝트에서 했던 작업을 Numpy 어레이 방식으로 바꿔보았습니다. 이를 통해 반복 연산이 얼마나 간단해질 수 있는지 느낄 수 있었습니다.



